In [ ]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook.display import (
    display_table,
    display_text,
)
from structural_tools.notebook.calculation import (
    check_value,
    set_params_columns,
)
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
%%render params
L_floor = 204 + (10.25 / 12) * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = L_floor
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 13.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

<IPython.core.display.Latex object>

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [7]:
%%render params
L_jog = 5 * ft

<IPython.core.display.Latex object>

In [8]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [9]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [10]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [11]:
set_params_columns(3)

In [12]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

<IPython.core.display.Latex object>

The roof weight is

In [13]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [14]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [15]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [16]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [17]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [18]:
set_params_columns(5)

In [19]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [20]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [21]:
set_params_columns(1)

In [22]:
%%render params
h_top__roof = (60 + 3 / 12) * ft
h_bottom__roof = (48) * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [23]:
%%render
h_n__check = check_value(h_n, 65 * ft, "<=")

<IPython.core.display.Latex object>

We will approximate the building period.

In [24]:
h_n = float(h_n)  # to prevent weirdness with units

In [25]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [26]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [27]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [28]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [29]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [30]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [31]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [32]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [33]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [ ]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Level 1", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [35]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [36]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,376.353229,44.5,16747.718698,0.296757,62.920082,0.000000,0.000000
Level 4,10.0,548.489146,34.5,18922.875531,0.335299,71.092004,62.920082,629.200824
Level 3,10.5,550.150271,24.0,13203.606500,0.233958,49.605085,134.012087,2036.327733
Level 2,10.5,560.117021,13.5,7561.579781,0.133986,28.408360,183.617171,3964.308033
Ground,13.5,0.000000,0.0,0.000000,0.000000,0.000000,212.025532,6826.652709


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [ ]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Level 4", "Level 3", "Level 2", "Level 1"]
walls = {
    "D-2": "N-S",
    # "D-2.7": "N-S",
    "D-2.8": "N-S",
    "D-3.2": "N-S",
    "D-4": "N-S",
    "D-5": "N-S",
    "D-6": "N-S",
    "D-6.2": "N-S",
    "D-6.7": "N-S",
    "D-6.8": "N-S",
    "D-7": "N-S",
    # E-W
    "D-B--D-1.8": "E-W",
    "D-B--D-3": "E-W",
    "D-B--D-3.7": "E-W",
    "D-B--D-4.5": "E-W",
    "D-B--D-5.8": "E-W",
    "D-B--D-6.5": "E-W",
    "D-B--D-7.2": "E-W",
    "D-C--D-1.2": "E-W",
    "D-C--D-1.8": "E-W",
    "D-C--D-3.2": "E-W",
    "D-C--D-3.6": "E-W",
    "D-C--D-3.9": "E-W",
    "D-C--D-5.2": "E-W",
    "D-C--D-6.7": "E-W",
    # "D-C.4--D-6.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [38]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [39]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [40]:
# Plan dimensions
l_D1__D2 = (35 + 1.25 / 12) * ft
l_D2__D3 = (23 + 8 / 12) * ft
l_D3__D4 = (34 + 4 / 12) * ft
l_D4__D5 = (24 + 8 / 12) * ft
l_D5__D6 = (18 + 7 / 12) * ft
l_D6__D7 = (37 + 1 / 12) * ft
l_D7__D8 = (36 + 5 / 12) * ft

l_D1__D21 = (36 + 5.25 / 12) * ft
l_D21__D28 = (17 + 11 / 12) * ft
l_D28__D36 = (26 + 4 / 12) * ft
l_D36__D43 = (21 + 4 / 12) * ft
l_D43__D6 = (34 + 4 / 12) * ft
l_D6__D67 = (26 + 1 / 12) * ft
l_D67__D73 = (21 + 4 / 12) * ft
l_D73__D8 = (26 + 1 / 12) * ft

l_D2__D27 = (18 + 8.75 / 12) * ft
l_D2__D28 = l_D1__D21 + l_D21__D28 - l_D1__D2
l_D28__D32 = (8 + 9.75 / 12) * ft
l_D32__D4 = l_D1__D2 + l_D2__D3 + l_D3__D4 - (l_D1__D21 + l_D21__D28 + l_D28__D32)
l_D6__D62 = (7 + 2 / 12) * ft
l_D6__D68 = (28 + 2.5 / 12) * ft
l_D62__D67 = l_D6__D67 - l_D6__D62
l_D62__D68 = l_D6__D68 - l_D6__D62
l_D67__D7 = l_D6__D7 - l_D6__D67
l_D68__D7 = l_D6__D7 - l_D6__D68
l_D67__D68 = l_D6__D68 - l_D6__D67

l_DA__DB = 30 + 9 / 12
l_DB__DC = 5 + 6 / 12
l_DC__DD = 30 + 9 / 12

d_trib_ns = 67 / 2 * ft

In [41]:
# Wall tributary widths and lengths

# Average of the top half and bottom half as we will use the full building depth for the trib area for this wall
t_w__D2_left = np.average(
    [l_D1__D2, l_D1__D2 - L_jog], weights=[(l_DA__DB + l_DB__DC) / (2 * d_trib_ns), l_DC__DD / (2 * d_trib_ns)]
)
t_w__D2_right = l_D2__D28 / 2
t_w__D2 = t_w__D2_left + t_w__D2_right

# t_w__D27 = (l_D4__D5 - L_jog) + ((l_D5__D6 + l_D6__D7) / 2)
t_w__D28 = (l_D2__D28 / 2) + (l_D28__D32 / 2)
t_w__D32 = (l_D28__D32 / 2) + (l_D32__D4 / 2)
t_w__D4 = (l_D32__D4 / 2) + (l_D4__D5 / 2)
t_w__D5 = (l_D4__D5 / 2) + (l_D5__D6 / 2)
t_w__D6 = (l_D5__D6 / 2) + (l_D6__D62 / 2)
t_w__D62 = (l_D6__D62 / 2) + (l_D62__D67 / 2)
t_w__D67 = (l_D62__D67 / 2) + (l_D67__D68 / 2)
t_w__D68 = (l_D67__D68 / 2) + (l_D68__D7 / 2)

t_w__D7_left = l_D68__D7 / 2
# Average of the top half and bottom half as we will use the full building depth for the trib area for this wall
t_w__D7_right = np.average(
    [l_D7__D8 - L_jog, l_D7__D8], weights=[(l_DA__DB + l_DB__DC) / (2 * d_trib_ns), l_DC__DD / (2 * d_trib_ns)]
)
t_w__D7 = t_w__D7_left + t_w__D7_right

t_l__D2 = float(2 * d_trib_ns)
t_l__D28 = float(2 * d_trib_ns)
t_l__D32 = float(2 * d_trib_ns)
t_l__D4 = float(2 * d_trib_ns)
t_l__D5 = float(2 * d_trib_ns)
t_l__D6 = float(2 * d_trib_ns)
t_l__D62 = float(2 * d_trib_ns)
t_l__D67 = float(2 * d_trib_ns)
t_l__D68 = float(2 * d_trib_ns)
t_l__D7 = float(2 * d_trib_ns)

In [ ]:
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
l_da__db = 29
l_dc__dd = 29
l_stairwall = 20
l_elevator = 10 + 7 / 12
wall_tributary = {
    "D-2": {"trib_width": float(t_w__D2), "trib_depth": t_l__D2, "wall_length": l_da__db},
    # "D-2.7": {"trib_width": float(t_w__D27), "trib_depth": t_l__D27, "wall_length": l_dc__dd},
    "D-2.8": {"trib_width": float(t_w__D28), "trib_depth": t_l__D28, "wall_length": l_dc__dd},
    "D-3.2": {"trib_width": float(t_w__D32), "trib_depth": t_l__D32, "wall_length": l_stairwall},
    "D-4": {"trib_width": float(t_w__D4), "trib_depth": t_l__D4, "wall_length": l_da__db},
    "D-5": {"trib_width": float(t_w__D5), "trib_depth": t_l__D5, "wall_length": l_da__db},
    "D-6": {"trib_width": float(t_w__D6), "trib_depth": t_l__D6, "wall_length": l_da__db},
    "D-6.2": {"trib_width": float(t_w__D62), "trib_depth": t_l__D62, "wall_length": l_elevator},
    "D-6.7": {"trib_width": float(t_w__D67), "trib_depth": t_l__D67, "wall_length": l_elevator},
    "D-6.8": {"trib_width": float(t_w__D68), "trib_depth": t_l__D68, "wall_length": l_stairwall},
    "D-7": {"trib_width": float(t_w__D7), "trib_depth": t_l__D7, "wall_length": l_da__db},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [43]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [44]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [45]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [46]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [47]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [48]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [ ]:
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 1",
    label="tab:shear_wall_forces_level1_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-2,4.584265,2843.104167,13033.541824,13033.541824,29.000000,449.432477,314.602734,460.714286,1,4.0
D-2.8,4.584265,940.093750,4309.638512,4309.638512,29.000000,148.608225,104.025757,310.714286,1,6.0
D-3.2,4.584265,1298.125000,5950.948502,5950.948502,20.000000,297.547425,208.283198,310.714286,1,6.0
D-4,4.584265,1829.239583,8385.718292,8385.718292,29.000000,289.162700,202.413890,310.714286,1,6.0
D-5,4.584265,1448.875000,6642.026392,6642.026392,29.000000,229.035393,160.324775,310.714286,1,6.0
D-6,4.584265,862.625000,3954.501262,3954.501262,29.000000,136.362112,95.453479,310.714286,1,6.0
D-6.2,4.584265,873.791667,4005.692217,4005.692217,10.583333,378.490603,264.943422,310.714286,1,6.0
D-6.7,4.584265,704.895833,3231.429025,3231.429025,10.583333,305.331876,213.732313,310.714286,1,6.0
D-6.8,4.584265,368.500000,1689.301510,1689.301510,20.000000,84.465076,59.125553,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-2,5.179659,2843.104167,14726.309542,27759.851367,29.000000,957.236254,670.065378,769.642857,1,2.0
D-2.8,5.179659,940.093750,4869.364874,9179.003385,29.000000,316.517358,221.562151,310.714286,1,6.0
D-3.2,5.179659,1298.125000,6723.844592,12674.793093,20.000000,633.739655,443.617758,460.714286,1,4.0
D-4,5.179659,1829.239583,9474.836922,17860.555214,29.000000,615.881214,431.116850,460.714286,1,4.0
D-5,5.179659,1448.875000,7504.678157,14146.704549,29.000000,487.817398,341.472179,460.714286,1,4.0
D-6,5.179659,862.625000,4468.103180,8422.604443,29.000000,290.434636,203.304245,310.714286,1,6.0
D-6.2,5.179659,873.791667,4525.942704,8531.634921,10.583333,806.138733,564.297113,600.000000,1,3.0
D-6.7,5.179659,704.895833,3651.119913,6882.548938,10.583333,650.319585,455.223709,460.714286,1,4.0
D-6.8,5.179659,368.500000,1908.704271,3598.005781,20.000000,179.900289,125.930202,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-2,3.614153,2843.104167,10275.414858,38035.266225,29.000000,1311.560904,918.092633,921.428571,2,4.0
D-2.8,3.614153,940.093750,3397.643111,12576.646497,29.000000,433.677465,303.574226,310.714286,1,6.0
D-3.2,3.614153,1298.125000,4691.623005,17366.416098,20.000000,868.320805,607.824563,769.642857,1,2.0
D-4,3.614153,1829.239583,6611.152632,24471.707845,29.000000,843.851995,590.696396,600.000000,1,3.0
D-5,3.614153,1448.875000,5236.456644,19383.161193,29.000000,668.384869,467.869408,600.000000,1,3.0
D-6,3.614153,862.625000,3117.659158,11540.263601,29.000000,397.940124,278.558087,310.714286,1,6.0
D-6.2,3.614153,873.791667,3158.017205,11689.652126,10.583333,1104.534059,773.173841,921.428571,2,4.0
D-6.7,3.614153,704.895833,2547.601739,9430.150677,10.583333,891.037859,623.726501,769.642857,1,2.0
D-6.8,3.614153,368.500000,1331.815563,4929.821344,20.000000,246.491067,172.543747,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-2,2.069791,2843.104167,5884.632296,43919.898521,29.000000,1514.479259,1060.135482,1200.000000,2,3.0
D-2.8,2.069791,940.093750,1945.797874,14522.444371,29.000000,500.773944,350.541761,460.714286,1,4.0
D-3.2,2.069791,1298.125000,2686.847844,20053.263942,20.000000,1002.663197,701.864238,769.642857,1,2.0
D-4,2.069791,1829.239583,3786.144193,28257.852038,29.000000,974.408691,682.086084,769.642857,1,2.0
D-5,2.069791,1448.875000,2998.868884,22382.030077,29.000000,771.794141,540.255898,600.000000,1,3.0
D-6,2.069791,862.625000,1785.453728,13325.717329,29.000000,459.507494,321.655246,460.714286,1,4.0
D-6.2,2.069791,873.791667,1808.566398,13498.218524,10.583333,1275.422223,892.795556,921.428571,2,4.0
D-6.7,2.069791,704.895833,1458.987270,10889.137947,10.583333,1028.894924,720.226447,769.642857,1,2.0
D-6.8,2.069791,368.500000,762.718098,5692.539441,20.000000,284.626972,199.238880,310.714286,1,6.0


#### East-West Direction

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

In [50]:
# Plan dimensions
l_DA__DB = (30 + 9 / 12) * ft
l_DB__DC = (5 + 6 / 12) * ft
l_DC__DD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [51]:
# Wall tributary widths and lengths
t_w__DB = l_DA__DB + l_DB__DC / 2
t_w__BC = l_DB__DC / 2 + l_DC__DD

t_l__DB__1 = 46.5
t_l__DB__2 = 28.25
t_l__DB__3 = 27.67
t_l__DB__4 = 27.75
t_l__DB__5 = 23
t_l__DB__6 = 18.33 + 1.24 / 12  # add 1.24 inches to pencil out 204'-10.25" length
t_l__DB__7 = 33.25

t_l__DC__1 = 37
t_l__DC__2 = 21.66
t_l__DC__3 = 16
t_l__DC__4 = 11.66
t_l__DC__5 = 33.67
t_l__DC__6 = 33.66 + 2.5 / 12  # add 2.5 inches to pencil out 204'-10.25" length
t_l__DC__7 = 51
# t_l__BC4__1 = 19

In [52]:
# Wall segment lengths
l_DB__1 = 9.67
l_DB__2 = 13.5
l_DB__3 = 16.75
l_DB__4 = 22.75
l_DB__5 = 18.25
l_DB__6 = 8.75
l_DB__7 = 14.67

l_DC__1 = 23.5
l_DC__2 = 10.25
l_DC__3 = 9.25
l_DC__4 = 6.75
l_DC__5 = 28.75
l_DC__6 = 14
l_DC__7 = 14.5
# l_DC4__1 = 19

In [ ]:
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "D-B--D-1.8": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__1, "wall_length": l_DB__1},
    "D-B--D-3": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__2, "wall_length": l_DB__2},
    "D-B--D-3.7": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__3, "wall_length": l_DB__3},
    "D-B--D-4.5": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__4, "wall_length": l_DB__4},
    "D-B--D-5.8": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__5, "wall_length": l_DB__5},
    "D-B--D-6.5": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__6, "wall_length": l_DB__6},
    "D-B--D-7.2": {"trib_width": float(t_w__DB), "trib_depth": t_l__DB__7, "wall_length": l_DB__7},
    "D-C--D-1.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__1, "wall_length": l_DC__1},
    "D-C--D-1.8": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__2, "wall_length": l_DC__2},
    "D-C--D-3.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__3, "wall_length": l_DC__3},
    "D-C--D-3.6": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__4, "wall_length": l_DC__4},
    "D-C--D-3.9": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__5, "wall_length": l_DC__5},
    "D-C--D-5.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__6, "wall_length": l_DC__6},
    "D-C--D-6.7": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC__7, "wall_length": l_DC__7},
    # "D-C.4--D-6.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__DC4__1, "wall_length": l_DC4__1},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew["aspect ratio factor"] = np.where(
    flexible_ew["aspect ratio"] > 2, 1.25 - 0.125 * flexible_ew["wall height"] / flexible_ew["wall length"], 1
)

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<=")

<IPython.core.display.Latex object>

In [55]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 1",
    label="tab:shear_wall_forces_level1_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-B--D-1.8,4.584265,1557.750000,2915.635047,2915.635047,9.67,301.513448,211.059414,310.714286,1,6.0
D-B--D-3,4.584265,946.375000,4070.431554,4070.431554,13.50,301.513448,211.059414,310.714286,1,6.0
D-B--D-3.7,4.584265,926.945000,5050.350262,5050.350262,16.75,301.513448,211.059414,310.714286,1,6.0
D-B--D-4.5,4.584265,929.625000,6859.430953,6859.430953,22.75,301.513448,211.059414,310.714286,1,6.0
D-B--D-5.8,4.584265,770.500000,5502.620435,5502.620435,18.25,301.513448,211.059414,310.714286,1,6.0
D-B--D-6.5,4.584265,617.516667,2638.242674,2638.242674,8.75,301.513448,211.059414,310.714286,1,6.0
D-B--D-7.2,4.584265,1113.875000,4423.202289,4423.202289,14.67,301.513448,211.059414,310.714286,1,6.0
D-C--D-1.2,4.584265,1239.500000,6909.588835,6909.588835,23.50,294.025057,205.817540,310.714286,1,6.0
D-C--D-1.8,4.584265,725.610000,3013.756832,3013.756832,10.25,294.025057,205.817540,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-B--D-1.8,5.179659,1557.750000,3294.311308,6209.946355,9.67,642.186800,449.530760,460.714286,1,4.0
D-B--D-3,5.179659,946.375000,4599.090243,8669.521798,13.50,642.186800,449.530760,460.714286,1,4.0
D-B--D-3.7,5.179659,926.945000,5706.278635,10756.628897,16.75,642.186800,449.530760,460.714286,1,4.0
D-B--D-4.5,5.179659,929.625000,7750.318744,14609.749697,22.75,642.186800,449.530760,460.714286,1,4.0
D-B--D-5.8,5.179659,770.500000,6217.288662,11719.909097,18.25,642.186800,449.530760,460.714286,1,4.0
D-B--D-6.5,5.179659,617.516667,2980.891824,5619.134499,8.75,642.186800,449.530760,460.714286,1,4.0
D-B--D-7.2,5.179659,1113.875000,4997.678065,9420.880354,14.67,642.186800,449.530760,460.714286,1,4.0
D-C--D-1.2,5.179659,1239.500000,7806.991021,14716.579856,23.50,626.237441,438.366208,460.714286,1,4.0
D-C--D-1.8,5.179659,725.610000,3405.176935,6418.933767,10.25,626.237441,438.366208,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-B--D-1.8,3.614153,1557.750000,2298.635328,8508.581682,9.67,879.894693,615.926285,769.642857,1,2.0
D-B--D-3,3.614153,946.375000,3209.056559,11878.578357,13.50,879.894693,615.926285,769.642857,1,2.0
D-B--D-3.7,3.614153,926.945000,3981.607212,14738.236109,16.75,879.894693,615.926285,769.642857,1,2.0
D-B--D-4.5,3.614153,929.625000,5407.854571,20017.604268,22.75,879.894693,615.926285,769.642857,1,2.0
D-B--D-5.8,3.614153,770.500000,4338.169052,16058.078149,18.25,879.894693,615.926285,769.642857,1,2.0
D-B--D-6.5,3.614153,617.516667,2079.944066,7699.078564,8.75,879.894693,615.926285,769.642857,1,2.0
D-B--D-7.2,3.614153,1113.875000,3487.174794,12908.055148,14.67,879.894693,615.926285,769.642857,1,2.0
D-C--D-1.2,3.614153,1239.500000,5447.398162,20163.978018,23.50,858.041618,600.629132,769.642857,1,2.0
D-C--D-1.8,3.614153,725.610000,2375.992815,8794.926582,10.25,858.041618,600.629132,769.642857,1,2.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
D-B--D-1.8,2.069791,1557.750000,1316.406576,9824.988258,9.67,1016.027741,711.219419,769.642857,1,2.0
D-B--D-3,2.069791,946.375000,1837.796150,13716.374507,13.50,1016.027741,711.219419,769.642857,1,2.0
D-B--D-3.7,2.069791,926.945000,2280.228556,17018.464666,16.75,1016.027741,711.219419,769.642857,1,2.0
D-B--D-4.5,2.069791,929.625000,3097.026845,23114.631113,22.75,1016.027741,711.219419,769.642857,1,2.0
D-B--D-5.8,2.069791,770.500000,2484.428129,18542.506277,18.25,1016.027741,711.219419,769.642857,1,2.0
D-B--D-6.5,2.069791,617.516667,1191.164171,8890.242736,8.75,1016.027741,711.219419,769.642857,1,2.0
D-B--D-7.2,2.069791,1113.875000,1997.071816,14905.126964,14.67,1016.027741,711.219419,769.642857,1,2.0
D-C--D-1.2,2.069791,1239.500000,3119.673083,23283.651102,23.50,990.793664,693.555565,769.642857,1,2.0
D-C--D-1.8,2.069791,725.610000,1360.708473,10155.635055,10.25,990.793664,693.555565,769.642857,1,2.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [60]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [61]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [ ]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["sheathed sides"] = flexible_ns["sheathed sides"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"]
    * rigid_ns["wall height"]
    / (1000 * rigid_ns["shear stiffness"] * rigid_ns["sheathed sides"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [63]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [64]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [65]:
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 1",
    label="tab:wall_rigidities_ns_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-2,13033.541824,0.357959,36410.733190,0.176964
D-2.8,4309.638512,0.194796,22123.874683,0.107527
D-3.2,5950.948502,0.344618,17268.229380,0.083927
D-4,8385.718292,0.297500,28187.303720,0.136996
D-5,6642.026392,0.253564,26194.627845,0.127311
D-6,3954.501262,0.185848,21278.202954,0.103416
D-6.2,4005.692217,0.523601,7650.278269,0.037182
D-6.7,3231.429025,0.468053,6903.982680,0.033555
D-6.8,1689.301510,0.187343,9017.145425,0.043825


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-2,27759.851367,0.545712,50869.037439,0.177980
D-2.8,9179.003385,0.333922,27488.480478,0.096176
D-3.2,12674.793093,0.540145,23465.543213,0.082101
D-4,17860.555214,0.482622,37007.374688,0.129481
D-5,14146.704549,0.401089,35270.740438,0.123405
D-6,8422.604443,0.313864,26835.205613,0.093891
D-6.2,8531.634921,0.735518,11599.487675,0.040584
D-6.7,6882.548938,0.683572,10068.500038,0.035228
D-6.8,3598.005781,0.271134,13270.226223,0.046430


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-2,38035.266225,0.925530,41095.662685,0.138709
D-2.8,12576.646497,0.424019,29660.576422,0.100113
D-3.2,17366.416098,0.551590,31484.260040,0.106268
D-4,24471.707845,0.572897,42715.722687,0.144178
D-5,19383.161193,0.472593,41014.499255,0.138435
D-6,11540.263601,0.396537,29102.642238,0.098230
D-6.2,11689.652126,0.987775,11834.324992,0.039944
D-6.7,9430.150677,0.701221,13448.190288,0.045391
D-6.8,4929.821344,0.322912,15266.754197,0.051530


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-2,43919.898521,1.253642,35033.843819,0.139487
D-2.8,14522.444371,0.534284,27181.126365,0.108222
D-3.2,20053.263942,0.816006,24574.895529,0.097845
D-4,28257.852038,0.727681,38832.736281,0.154613
D-5,22382.030077,0.695940,32160.860247,0.128049
D-6,13325.717329,0.499847,26659.617224,0.106145
D-6.2,13498.218524,1.472926,9164.220132,0.036487
D-6.7,10889.137947,1.036716,10503.491575,0.041820
D-6.8,5692.539441,0.459885,12378.172946,0.049284


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [ ]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["sheathed sides"] = flexible_ew["sheathed sides"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"]
    * rigid_ew["wall height"]
    / (1000 * rigid_ew["shear stiffness"] * rigid_ew["sheathed sides"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [67]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [69]:
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 1",
    label="tab:wall_rigidities_ew_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-B--D-1.8,2915.635047,0.488746,5965.541926,0.035611
D-B--D-3,4070.431554,0.411187,9899.214461,0.059093
D-B--D-3.7,5050.350262,0.373192,13532.836507,0.080784
D-B--D-4.5,6859.430953,0.331568,20687.857469,0.123496
D-B--D-5.8,5502.620435,0.360220,15275.707657,0.091188
D-B--D-6.5,2638.242674,0.517490,5098.152683,0.030433
D-B--D-7.2,4423.202289,0.395570,11181.852092,0.066750
D-C--D-1.2,6909.588835,0.322359,21434.464239,0.127953
D-C--D-1.8,3013.756832,0.467580,6445.435201,0.038476


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-B--D-1.8,6209.946355,0.704712,8812.040129,0.038029
D-B--D-3,8669.521798,0.617312,14043.990333,0.060608
D-B--D-3.7,10756.628897,0.574496,18723.596851,0.080804
D-B--D-4.5,14609.749697,0.527590,27691.478950,0.119505
D-B--D-5.8,11719.909097,0.559878,20932.970816,0.090338
D-B--D-6.5,5619.134499,0.737103,7623.273653,0.032899
D-B--D-7.2,9420.880354,0.599713,15708.990661,0.067794
D-C--D-1.2,14716.579856,0.513186,28676.905412,0.123758
D-C--D-1.8,6418.933767,0.676571,9487.455542,0.040944


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-B--D-1.8,8508.581682,0.723309,11763.417888,0.037878
D-B--D-3,11878.578357,0.632065,18793.299942,0.060515
D-B--D-3.7,14738.236109,0.587365,25092.116565,0.080797
D-B--D-4.5,20017.604268,0.538396,37180.062582,0.119721
D-B--D-5.8,16058.078149,0.572104,28068.444482,0.090381
D-B--D-6.5,7699.078564,0.757124,10168.841935,0.032744
D-B--D-7.2,12908.055148,0.613691,21033.469913,0.067728
D-C--D-1.2,20163.978018,0.523544,38514.372406,0.124017
D-C--D-1.8,8794.926582,0.693958,12673.569865,0.040809


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
D-B--D-1.8,9824.988258,1.068483,9195.272515,0.037917
D-B--D-3,13716.374507,0.934541,14677.127498,0.060522
D-B--D-3.7,17018.464666,0.868924,19585.667176,0.080762
D-B--D-4.5,23114.631113,0.797040,29000.576943,0.119585
D-B--D-5.8,18542.506277,0.846522,21904.335220,0.090323
D-B--D-6.5,8890.242736,1.118122,7951.045835,0.032786
D-B--D-7.2,14905.126964,0.907570,16423.121777,0.067721
D-C--D-1.2,23283.651102,0.774566,30060.236687,0.123955
D-C--D-1.8,10155.635055,1.024072,9916.915867,0.040893


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The building is symmetric in each direction when projected to that direction, so the centers of mass are just half of each total plan dimension.

In [70]:
%%render
X_plan = L_floor + L_jog
Y_plan = w_building
CM_x = X_plan / 2
CM_y = Y_plan / 2

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [71]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [72]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [73]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "D-2": {"x": float(l_D1__D2), "y": 0},
    "D-2.8": {"x": float(l_D1__D2 + l_D2__D28), "y": 0},
    "D-3.2": {"x": float(l_D1__D2 + l_D2__D28 + l_D28__D32), "y": 0},
    "D-4": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4), "y": 0},
    "D-5": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5), "y": 0},
    "D-6": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5 + l_D5__D6), "y": 0},
    "D-6.2": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5 + l_D5__D6 + l_D6__D62), "y": 0},
    "D-6.7": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5 + l_D5__D6 + l_D6__D67), "y": 0},
    "D-6.8": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5 + l_D5__D6 + l_D6__D68), "y": 0},
    "D-7": {"x": float(l_D1__D2 + l_D2__D3 + l_D3__D4 + l_D4__D5 + l_D5__D6 + l_D6__D7), "y": 0},
    # E-W
    "D-B--D-1.8": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-3": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-3.7": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-4.5": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-5.8": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-6.5": {"x": 0, "y": float(l_DA__DB)},
    "D-B--D-7.2": {"x": 0, "y": float(l_DA__DB)},
    "D-C--D-1.2": {"x": 0, "y": float(l_DA__DB)},
    "D-C--D-1.8": {"x": 0, "y": float(l_DA__DB)},
    "D-C--D-3.2": {"x": 0, "y": float(l_DA__DB + l_DB__DC)},
    "D-C--D-3.6": {"x": 0, "y": float(l_DA__DB + l_DB__DC)},
    "D-C--D-3.9": {"x": 0, "y": float(l_DA__DB + l_DB__DC)},
    "D-C--D-5.2": {"x": 0, "y": float(l_DA__DB + l_DB__DC)},
    "D-C--D-6.7": {"x": 0, "y": float(l_DA__DB + l_DB__DC)},
}

In [74]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_x = float(X_plan)
CMx = plan_dim_x / 2
CMxa_plus = CMx + plan_dim_x * 0.05
CMxa_minus = CMx - plan_dim_x * 0.05
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, 0, rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_y = float(w_building)
CMy = plan_dim_y / 2
CMya_plus = CMy + plan_dim_y * 0.05
CMya_minus = CMy - plan_dim_y * 0.05
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, 0, rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [75]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 1",
    label="tab:center_of_rigidity_level1",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,103.100389,67.996223,8.666014,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,103.100389,48.746223,8.666014,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,103.100389,39.933723,8.666014,0.000000,0.000000,0.000000
D-4,93.104167,0.00,103.100389,9.996223,8.666014,0.000000,0.000000,0.000000
D-5,117.770833,0.00,103.100389,14.670444,12.319402,0.000000,0.000000,0.000000
D-6,136.354167,0.00,103.100389,33.253777,12.319402,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,103.100389,40.420444,12.319402,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,103.100389,59.337111,12.319402,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,103.100389,61.462111,12.319402,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,105.5439,70.439734,11.109525,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,105.5439,51.189734,11.109525,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,105.5439,42.377234,11.109525,0.000000,0.000000,0.000000
D-4,93.104167,0.00,105.5439,12.439734,11.109525,0.000000,0.000000,0.000000
D-5,117.770833,0.00,105.5439,12.226933,9.875891,0.000000,0.000000,0.000000
D-6,136.354167,0.00,105.5439,30.810266,9.875891,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,105.5439,37.976933,9.875891,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,105.5439,56.893600,9.875891,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,105.5439,59.018600,9.875891,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,105.526521,70.422354,11.092146,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,105.526521,51.172354,11.092146,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,105.526521,42.359854,11.092146,0.000000,0.000000,0.000000
D-4,93.104167,0.00,105.526521,12.422354,11.092146,0.000000,0.000000,0.000000
D-5,117.770833,0.00,105.526521,12.244312,9.893271,0.000000,0.000000,0.000000
D-6,136.354167,0.00,105.526521,30.827646,9.893271,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,105.526521,37.994312,9.893271,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,105.526521,56.910979,9.893271,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,105.526521,59.035979,9.893271,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
D-2,35.104167,0.00,104.991038,69.886871,10.556663,0.000000,0.000000,0.000000
D-2.8,54.354167,0.00,104.991038,50.636871,10.556663,0.000000,0.000000,0.000000
D-3.2,63.166667,0.00,104.991038,41.824371,10.556663,0.000000,0.000000,0.000000
D-4,93.104167,0.00,104.991038,11.886871,10.556663,0.000000,0.000000,0.000000
D-5,117.770833,0.00,104.991038,12.779796,10.428754,0.000000,0.000000,0.000000
D-6,136.354167,0.00,104.991038,31.363129,10.428754,0.000000,0.000000,0.000000
D-6.2,143.520833,0.00,104.991038,38.529796,10.428754,0.000000,0.000000,0.000000
D-6.7,162.437500,0.00,104.991038,57.446462,10.428754,0.000000,0.000000,0.000000
D-6.8,164.562500,0.00,104.991038,59.571462,10.428754,0.000000,0.000000,0.000000


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [76]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [ ]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 1",
    label="tab:rigid_forces_level1",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,545266.336025,0.000000,67.996223,0.000000,0.176964,2674.850926,11134.569880,13809.420806,13033.541824
D-2.8,545266.336025,0.000000,48.746223,0.000000,0.107527,1165.165265,6765.582758,7930.748022,4309.638512
D-3.2,545266.336025,0.000000,39.933723,0.000000,0.083927,745.028657,5280.704064,6025.732721,5950.948502
D-4,545266.336025,0.000000,9.996223,0.000000,0.136996,304.421154,8619.807279,8924.228433,8385.718292
D-5,775137.809495,0.000000,14.670444,0.000000,0.127311,590.216162,8010.437820,8600.653982,6642.026392
D-6,775137.809495,0.000000,33.253777,0.000000,0.103416,1086.754728,6506.972449,7593.727177,3954.501262
D-6.2,775137.809495,0.000000,40.420444,0.000000,0.037182,474.934719,2339.490324,2814.425043,4005.692217
D-6.7,775137.809495,0.000000,59.337111,0.000000,0.033555,629.189831,2111.269696,2740.459527,3231.429025
D-6.8,775137.809495,0.000000,61.462111,0.000000,0.043825,851.201051,2757.484595,3608.685645,1689.301510


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,1.488811e+06,0.000000,70.439734,0.000000,0.177980,7300.469601,23851.478804,31151.948405,27759.851367
D-2.8,1.488811e+06,0.000000,51.189734,0.000000,0.096176,2866.904161,12888.801174,15755.705335,9179.003385
D-3.2,1.488811e+06,0.000000,42.377234,0.000000,0.082101,2026.015608,11002.525991,13028.541600,12674.793093
D-4,1.488811e+06,0.000000,12.439734,0.000000,0.129481,937.948360,17352.021139,18289.969499,17860.555214
D-5,1.323489e+06,0.000000,12.226933,0.000000,0.123405,781.074560,16537.747917,17318.822477,14146.704549
D-6,1.323489e+06,0.000000,30.810266,0.000000,0.093891,1497.479070,12582.493598,14079.972667,8422.604443
D-6.2,1.323489e+06,0.000000,37.976933,0.000000,0.040584,797.846009,5438.768814,6236.614823,8531.634921
D-6.7,1.323489e+06,0.000000,56.893600,0.000000,0.035228,1037.501108,4720.919194,5758.420302,6882.548938
D-6.8,1.323489e+06,0.000000,59.018600,0.000000,0.046430,1418.494342,6222.144853,7640.639195,3598.005781


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,2.036708e+06,0.000000,70.422354,0.000000,0.138709,8646.376536,25469.422508,34115.799044,38035.266225
D-2.8,2.036708e+06,0.000000,51.172354,0.000000,0.100113,4534.637775,18382.420512,22917.058288,12576.646497
D-3.2,2.036708e+06,0.000000,42.359854,0.000000,0.106268,3984.515908,19512.665544,23497.181452,17366.416098
D-4,2.036708e+06,0.000000,12.422354,0.000000,0.144178,1585.328576,26473.469893,28058.798469,24471.707845
D-5,1.816574e+06,0.000000,12.244312,0.000000,0.138435,1338.208367,25419.120710,26757.329077,19383.161193
D-6,1.816574e+06,0.000000,30.827646,0.000000,0.098230,2390.697940,18036.635567,20427.333507,11540.263601
D-6.2,1.816574e+06,0.000000,37.994312,0.000000,0.039944,1198.157770,7334.433943,8532.591713,11689.652126
D-6.7,1.816574e+06,0.000000,56.910979,0.000000,0.045391,2039.444215,8334.642102,10374.086317,9430.150677
D-6.8,1.816574e+06,0.000000,59.035979,0.000000,0.051530,2401.681492,9461.714146,11863.395637,4929.821344


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
D-2,2.238282e+06,0.000000,69.886871,0.000000,0.139487,9496.022815,29574.904001,39070.926817,43919.898521
D-2.8,2.238282e+06,0.000000,50.636871,0.000000,0.108222,5338.173373,22945.789422,28283.962795,14522.444371
D-3.2,2.238282e+06,0.000000,41.824371,0.000000,0.097845,3986.387041,20745.658966,24732.046007,20053.263942
D-4,2.238282e+06,0.000000,11.886871,0.000000,0.154613,1790.292159,32781.856697,34572.148857,28257.852038
D-5,2.211162e+06,0.000000,12.779796,0.000000,0.128049,1574.764790,27149.585964,28724.350754,22382.030077
D-6,2.211162e+06,0.000000,31.363129,0.000000,0.106145,3203.593657,22505.541333,25709.134991,13325.717329
D-6.2,2.211162e+06,0.000000,38.529796,0.000000,0.036487,1352.870884,7736.260173,9089.131057,13498.218524
D-6.7,2.211162e+06,0.000000,57.446462,0.000000,0.041820,2311.857773,8866.847629,11178.705402,10889.137947
D-6.8,2.211162e+06,0.000000,59.571462,0.000000,0.049284,2825.263373,10449.417954,13274.681327,5692.539441


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [ ]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    caption="Final shear wall design values - Level 1",
    label="tab:shear_walls_level1",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,13809.420806,29.000000,476.186924,333.330847,460.714286,1,4.0
D-2.8,7930.748022,29.000000,273.474070,191.431849,310.714286,1,6.0
D-3.2,6025.732721,20.000000,301.286636,210.900645,310.714286,1,6.0
D-4,8924.228433,29.000000,307.732015,215.412410,310.714286,1,6.0
D-5,8600.653982,29.000000,296.574275,207.601993,310.714286,1,6.0
D-6,7593.727177,29.000000,261.852661,183.296863,310.714286,1,6.0
D-6.2,4005.692217,10.583333,378.490603,264.943422,310.714286,1,6.0
D-6.7,3231.429025,10.583333,305.331876,213.732313,310.714286,1,6.0
D-6.8,3608.685645,20.000000,180.434282,126.303998,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,31151.948405,29.000000,1074.205117,751.943582,769.642857,1,2.0
D-2.8,15755.705335,29.000000,543.300184,380.310129,460.714286,1,4.0
D-3.2,13028.541600,20.000000,651.427080,455.998956,460.714286,1,4.0
D-4,18289.969499,29.000000,630.688603,441.482022,460.714286,1,4.0
D-5,17318.822477,29.000000,597.200775,418.040543,460.714286,1,4.0
D-6,14079.972667,29.000000,485.516299,339.861409,460.714286,1,4.0
D-6.2,8531.634921,10.583333,806.138733,564.297113,600.000000,1,3.0
D-6.7,6882.548938,10.583333,650.319585,455.223709,460.714286,1,4.0
D-6.8,7640.639195,20.000000,382.031960,267.422372,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,38035.266225,29.000000,1311.560904,918.092633,921.428571,2,4.0
D-2.8,22917.058288,29.000000,790.243389,553.170372,600.000000,1,3.0
D-3.2,23497.181452,20.000000,1174.859073,822.401351,921.428571,2,4.0
D-4,28058.798469,29.000000,967.544775,677.281342,769.642857,1,2.0
D-5,26757.329077,29.000000,922.666520,645.866564,769.642857,1,2.0
D-6,20427.333507,29.000000,704.390811,493.073567,600.000000,1,3.0
D-6.2,11689.652126,10.583333,1104.534059,773.173841,921.428571,2,4.0
D-6.7,10374.086317,10.583333,980.228628,686.160040,769.642857,1,2.0
D-6.8,11863.395637,20.000000,593.169782,415.218847,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
D-2,43919.898521,29.000000,1514.479259,1060.135482,1200.000000,2,3.0
D-2.8,28283.962795,29.000000,975.309062,682.716343,769.642857,1,2.0
D-3.2,24732.046007,20.000000,1236.602300,865.621610,921.428571,2,4.0
D-4,34572.148857,29.000000,1192.143064,834.500145,921.428571,2,4.0
D-5,28724.350754,29.000000,990.494854,693.346398,769.642857,1,2.0
D-6,25709.134991,29.000000,886.521896,620.565327,769.642857,1,2.0
D-6.2,13498.218524,10.583333,1275.422223,892.795556,921.428571,2,4.0
D-6.7,11178.705402,10.583333,1056.255629,739.378940,769.642857,1,2.0
D-6.8,13274.681327,20.000000,663.734066,464.613846,600.000000,1,3.0


### Shear Wall Axial Forces

#### Tension Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D - 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-10)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall height"] = rigid["wall height"]
shear_walls["tributary area"] = flexible["tributary area"]
shear_walls["floor OTM"] = shear_walls["shear force demand"] * shear_walls["wall height"]
shear_walls["OTM"] = shear_walls.groupby(level="Wall")["floor OTM"].cumsum()
shear_walls["dead load"] = 0.6 * float(U_floor * 1000) * shear_walls["tributary area"]

# Reduce resisting moment dead load in some walls that definitely are not correct
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-2", "dead load"] *= 1 / 8
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-2.1", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-2.8", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-3", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-3.2", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-4", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-5", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-6", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-6.2", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-6.7", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-6.8", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "D-7", "dead load"] *= 1 / 16
shear_walls.loc[shear_walls["line"] == "D-B", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "D-C", "dead load"] *= 0

shear_walls["RM"] = shear_walls["dead load"] * shear_walls["wall length"] / 2

shear_walls["total moment"] = shear_walls["OTM"] - shear_walls["RM"]
shear_walls["total moment"] = np.where(
    shear_walls["RM"] <= shear_walls["OTM"], shear_walls["OTM"] - shear_walls["RM"], 0
)
wall_arm_shortening_length = 0.5  # feet due to holddown positioning
shear_walls["tension force"] = shear_walls["total moment"] / (shear_walls["wall length"] - wall_arm_shortening_length)
num_bins = 5
shear_walls["binned tension force"] = (
    shear_walls["tension force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else 0))
)

#### Compression Forces

We will calculate the compression forces in a very similar way. However, we will ignore the resisting moment.

The live load, $L$, is $40$ psf, while the snow load, $S$, is $25$ psf and applies only to corridor walls as that is how the roof trusses are supported.

$$
C_{seismic} = \frac{M_{OT}}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [81]:
shear_walls["compression force"] = shear_walls["OTM"] / (shear_walls["wall length"] - wall_arm_shortening_length)
shear_walls["binned compression force"] = (
    shear_walls["compression force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins).map(lambda x: x.right if pd.notna(x) else 0))
    # .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else pd.NA))
)

#### Holddown Forces Table

With the tension and compression forces calculated, we can construct a table of the forces for each level. We will default to 5 bins for each level to give us options A through E.

However, there is only one table in the plans for all the holddown forces, so these will all get compared together as an aggregate of all buildings and simplified further.

In [94]:
NUM_OPTIONS = 5
LABELS = list("ABCDE")

schedule_rows = []


def assign_schedule(group):
    group = group.copy()

    level = group.index.get_level_values("Level")[0]

    n = len(group)
    group["_bin"] = np.floor(np.arange(n) * min(NUM_OPTIONS, n) / n)

    schedule = (
        group.groupby("_bin")
        .agg(
            tension=("tension force", "max"),
            compression=("compression force", "max"),
        )
        .reset_index()
    )

    # convert to kips
    schedule["tension"] /= 1000
    schedule["compression"] /= 1000

    # sort ONLY schedule
    schedule["governing"] = schedule[["tension", "compression"]].max(axis=1)
    schedule = schedule.sort_values("governing").reset_index(drop=True)

    # assign A–E
    schedule["option"] = LABELS[: len(schedule)]

    # IMPORTANT: keep level in schedule
    schedule["Level"] = level

    # map bins → options
    bin_to_option = dict(zip(schedule["_bin"], schedule["option"]))
    group["option"] = group["_bin"].map(bin_to_option)

    schedule_map = schedule.set_index("option")[["tension", "compression"]]

    group["binned tension force"] = group["option"].map(schedule_map["tension"])
    group["binned compression force"] = group["option"].map(schedule_map["compression"])

    schedule_rows.append(schedule[["Level", "option", "tension", "compression"]])

    return group.drop(columns=["_bin"])


shear_walls = shear_walls.groupby(level="Level", group_keys=False).apply(assign_schedule)

schedule_df = (
    pd.concat(schedule_rows, ignore_index=True)
    .rename(
        columns={
            "tension": "binned tension force",
            "compression": "binned compression force",
        }
    )
    .set_index(["Level", "option"])
    .sort_index()
)

In [ ]:
column_name_map = {
    "binned tension force": "$T$ [kip]",
    "binned compression force": "$C$ [kip]",
}
display_table(
    schedule_df,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Holddown Schedule",
    label="tab:holddown_schedule",
)

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [83]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [84]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,376.353229,376.353229,62.920082,62.920082,50.972900,101.945800,62.920082
Level 4,548.489146,924.842375,134.012087,79.477516,74.286814,148.573628,79.477516
Level 3,550.150271,1474.992646,183.617171,68.486468,74.511795,149.023590,74.511795
Level 2,560.117021,2035.109667,212.025532,58.355140,75.861682,151.723363,75.861682


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [ ]:
# Set up diaphragm dataframe
levels = ["Level 4", "Level 3", "Level 2", "Level 1"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [86]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 30
diaphragms_ns["L_cant"] = 35
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [87]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [88]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 31
diaphragms_ew["depth"] = float(L_floor)
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


In [89]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [90]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [91]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          62920.082384   4.584265       4607.185937   
        E-W          62920.082384   4.584265       2582.540695   
Level 4 N-S          79477.516242   5.790615       5819.567954   
        E-W          79477.516242   5.790615       3262.136861   
Level 3 N-S          74511.795196   5.428820       5455.963850   
        E-W          74511.795196   5.428820       3058.319952   
Level 2 N-S          75861.681716   5.527170       5554.806350   
        E-W          75861.681716   5.527170       3113.725742   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            10750.100519         68.763969        160.449261  
        E-W            29112.276924         12.606728        142.112203  
Level 4 N-S            13578.991893         86.859223        202.671521  
        E-W            36773.179157         15.924191        179.509061  
Level 3 N-S            12730.582318         81.432296        190.008691  
        E-W            34475.606733         14.929254        168.293412  
Level 2 N-S            12961.214816         82.907557        193.450967  
        E-W            35100.181093         15.199719        171.342285

In [92]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               68.763969            189.285714          0.363281   
        E-W               12.606728            189.285714          0.066602   
Level 4 N-S               86.859223            189.285714          0.458879   
        E-W               15.924191            189.285714          0.084128   
Level 3 N-S               81.432296            189.285714          0.430208   
        E-W               14.929254            189.285714          0.078872   
Level 2 N-S               82.907557            189.285714          0.438002   
        E-W               15.199719            189.285714          0.080300   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              160.449261            189.285714          0.847656  
        E-W              142.112203            189.285714          0.750781  
Level 4 N-S              202.671521            214.285714          0.945800  
        E-W              179.509061            189.285714          0.948350  
Level 3 N-S              190.008691            214.285714          0.886707  
        E-W              168.293412            189.285714          0.889097  
Level 2 N-S              193.450967            214.285714          0.902771  
        E-W              171.342285            189.285714          0.905205

In [93]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'

## Story Drift Determination

The inelastic design story drift is calculated as the difference in deflections between stories at their centers of mass.

We can find the shear wall line that is closest to the center of mass, then take the cumulative force on that line and divide it by the cumulative wall stiffness on that line. This will give us an approximate elastic story drift that will then be converted to inelastic story drift with $C_d$ and $I_e$.

Note that this is a conservative estimation of inelastic story drift due to assuming the deflection of a single shear wall line, rather than the full deflection of the diaphragm as a rigid unit.

\begin{align*}
    \delta_{xe} &= F/k \\
    \delta_x &= \frac{C_d \delta_{xe}}{I_e}
\end{align*}

In [ ]:
%%render
C_d
I_e

<IPython.core.display.Latex object>

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall stiffness"] = rigid["wall stiffness"]
shear_walls["cumulative shear demand"] = flexible["cumulative shear demand"]
shear_walls["x"] = rigid["x"]
shear_walls["y"] = rigid["y"]
shear_walls["delta_sw"] = rigid["delta_sw"]


def line_deflection(group, line):
    line_walls = group[group["line"] == line]

    total_stiffness = line_walls["wall stiffness"].sum()
    total_shear = line_walls["cumulative shear demand"].sum()

    return total_shear / total_stiffness


def get_cm_line_deflection(group, CM_x, CM_y, C_d, I_e):
    x_wall = group.loc[(group["x"] - CM_x).abs().idxmin()]

    y_wall = group.loc[(group["y"] - CM_y).abs().idxmin()]

    x_line = x_wall["line"]
    y_line = y_wall["line"]

    x_delta_sw = line_deflection(group, x_line)
    y_delta_sw = line_deflection(group, y_line)

    drift_factor = C_d / I_e

    return pd.Series(
        {
            "x line": x_line,
            "x delta_sw": x_delta_sw,
            "x inelastic drift": x_delta_sw * drift_factor,
            "y line": y_line,
            "y delta_sw": y_delta_sw,
            "y inelastic drift": y_delta_sw * drift_factor,
        }
    )


story_drift = shear_walls.groupby(level="Level", sort=False).apply(
    get_cm_line_deflection,
    CM_x=CM_x,
    CM_y=CM_y,
    C_d=C_d,
    I_e=I_e,
)
story_drift["allowable story drift"] = seismic_loads["floor height"] * 0.02 * 12

In [ ]:
column_name_map = {
    "x line": "N-S line",
    "x delta_sw": r"$\delta_{xe,ns}$ [in]",
    "x inelastic drift": r"$\delta_{x,ns}$ [in]",
    "y line": "E-W line",
    "y delta_sw": r"$\delta_{xe,ew}$ [in]",
    "y inelastic drift": r"$\delta_{x,ew}$ [in]",
    "allowable story drift": r"$\Delta_a$ [in]",
}
display_table(
    story_drift,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Story Drifts",
    label="tab:story_drifts",
)

,N-S line,"$\delta_{sw,ns}$ [in]","$\delta_{x,ns}$ [in]",E-W line,"$\delta_{sw,ew}$ [in]","$\delta_{x,ew}$ [in]",$\Delta_a$ [in]
Level,,,,,,,
Level 4,A-6,0.197989,0.791958,A-B,0.374226,1.496904,2.40
Level 3,A-6,0.341671,1.366682,A-B,0.565314,2.261256,2.52
Level 2,A-6,0.373059,1.492237,A-B,0.653255,2.613021,2.52
Level 1,A-6,0.453864,1.815455,A-B,0.697459,2.789837,2.76


In [ ]:
# Save dataframes to CSV
shear_walls.to_csv("shear_walls_D.csv")